# IVR: Iterative Visual Retracing
## 基于迭代视觉追溯的 VLM 幻觉缓解 —— 医学影像诊断

Kaggle 2x T4 | MiniCPM-V-4.6-int4 (1.3B) | 成本: 零

## 1. 环境安装

In [ ]:
!pip install -q "transformers[torch]>=5.7.0" torchvision av pyyaml rouge-score matplotlib Pillow

## 2. 拉取项目代码

从 GitHub 拉取 IVR 项目代码，无需手动上传

In [ ]:
import os

REPO_URL = "https://github.com/JKpink/yyz-project.git"
REPO_DIR = "/kaggle/working/yyz-project"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo exists, pulling latest changes...")
    !cd {REPO_DIR} && git pull origin main

import sys
sys.path.insert(0, os.path.join(REPO_DIR, "src"))

# 检查项目文件
!echo "=== 项目结构 ===" && ls -R {REPO_DIR}/src/ {REPO_DIR}/configs/

## 3. 加载模型

In [ ]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

# 两个模型
BASE_MODEL = "openbmb/MiniCPM-V-4.6"           # B1/B3/B4 用
THINKING_MODEL = "openbmb/MiniCPM-V-4.6-Thinking"  # B2 用（内置推理链）

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# B1/B3/B4 用的标准模型
print(f"\nLoading base model: {BASE_MODEL}")
base_model = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto",
)
base_processor = AutoProcessor.from_pretrained(BASE_MODEL, trust_remote_code=True)
print(f"Base model VRAM: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

print(f"\nVRAM before thinking model: {torch.cuda.memory_allocated() / 1e9:.1f} GB")
print(f"Free VRAM: {torch.cuda.memory_reserved() / 1e9:.1f} GB")
print("SKIP thinking model for now - load only when needed for B2")

## 4. 初始化 Baseline

In [ ]:
from ivr import IVRInference
from baselines.baseline_direct import BaselineDirect
from baselines.baseline_cot import BaselineThinking
from baselines.baseline_memvr import BaselineMemVR
from utils.config import get_config

# 加载配置
CONFIG_DIR = os.path.join(REPO_DIR, "configs")
config = get_config(CONFIG_DIR)

# B1: 标准模型, 直接回答
b1 = BaselineDirect(base_model, base_processor)

# B2: Thinking 模型 (需要时加载——两个模型同时驻留可能 OOM)
# b2 = BaselineThinking(thinking_model, thinking_processor)

# B3: 标准模型 + MemVR 追溯
b3 = BaselineMemVR(base_model, base_processor)

# B4: 标准模型 + IVR 迭代追溯
b4 = IVRInference(base_model, base_processor, config)

# 评测时逐个运行, B2 单独跑:
#   1. 先跑 B1/B3/B4 (共用 base_model)
#   2. del base_model; torch.cuda.empty_cache()
#   3. 加载 Thinking 模型跑 B2
baselines = {
    "B1_Direct": b1,
    "B3_MemVR": b3,
    "B4_IVR": b4,
}
print("B1/B3/B4 initialized (sharing base_model).")
print("B2 (Thinking model) will be run separately to avoid OOM.")

## 5. 加载数据

IU X-Ray 数据集需从 [Open-i](https://openi.nlm.nih.gov/) 下载后上传到 Kaggle Dataset。

如果暂无医学数据，可先用 VQA v2 子集做功能验证。

In [ ]:
from PIL import Image
from pathlib import Path
import json

def load_images(data_dir: str, max_images: int = 100):
    """从目录加载图像"""
    data_path = Path(data_dir)
    files = sorted(data_path.glob("*.png")) + sorted(data_path.glob("*.jpg"))
    images = []
    for f in files[:max_images]:
        try:
            img = Image.open(f).convert("RGB")
            images.append((f.name, img))
        except Exception as e:
            print(f"Error loading {f}: {e}")
    print(f"Loaded {len(images)} images from {data_dir}")
    return images

# 修改为你的数据路径
# DATA_DIR = "/kaggle/input/iu-xray/images"
# 先用 VQA v2 验证功能：
# DATA_DIR = "/kaggle/input/vqa-v2/val2014"

DATA_DIR = "/kaggle/input/iu-xray/images"  # 改成实际路径
MAX_IMAGES = 100  # 功能验证用 100 张，全量评测改为 7470

images = load_images(DATA_DIR, MAX_IMAGES)

# 医学诊断问题
QUESTION = "请描述这张胸片中的异常发现。如果未发现异常，请说明是正常胸片。"

## 6. 运行评测

In [ ]:
import time
from collections import defaultdict

results = {}
timing = {}

for name, baseline in baselines.items():
    print(f"\n{'='*50}")
    print(f"Running: {name}")
    print(f"{'='*50}")
    
    baseline_results = []
    start = time.time()
    
    for i, (fname, img) in enumerate(images):
        try:
            result = baseline.generate(img, QUESTION)
            result["image"] = fname
            result["baseline"] = name
            baseline_results.append(result)
        except Exception as e:
            print(f"  Error on {fname}: {e}")
            continue
        
        if (i + 1) % 20 == 0:
            elapsed = time.time() - start
            print(f"  [{name}] {i+1}/{len(images)} | {elapsed:.1f}s")
    
    elapsed = time.time() - start
    avg_passes = sum(r.get("num_passes", 1) for r in baseline_results) / len(baseline_results) if baseline_results else 0
    
    results[name] = baseline_results
    timing[name] = {
        "total_seconds": round(elapsed, 1),
        "seconds_per_image": round(elapsed / len(images), 2) if images else 0,
        "avg_passes": round(avg_passes, 2),
    }
    
    print(f"  [{name}] Done. {elapsed:.1f}s total, "
          f"{elapsed/len(images):.2f}s/img, "
          f"avg {avg_passes:.1f} passes" if images else "")

print("\n" + "="*50)
print("All baselines complete!")
print(json.dumps(timing, indent=2, ensure_ascii=False))

## 7. 结果汇总

In [ ]:
import pandas as pd

summary_rows = []
for name, timing_info in timing.items():
    summary_rows.append({
        "Baseline": name,
        **timing_info,
        "num_images": len(results[name]) if name in results else 0,
    })

df = pd.DataFrame(summary_rows)
df = df.sort_values("avg_passes")
print("\n评测结果汇总:")
print(df.to_string(index=False))

## 8. 查看示例输出

对比各 baseline 在同一张图上的表现

In [ ]:
from IPython.display import display, Markdown

# 取第一张图的结果
sample_idx = 0
if all(name in results for name in ["B1_Direct", "B4_IVR"]):
    img_name = results["B1_Direct"][sample_idx]["image"]
    print(f"示例图像: {img_name}\n")
    
    for name in ["B1_Direct", "B2_CoT", "B3_MemVR", "B4_IVR"]:
        if name in results and sample_idx < len(results[name]):
            r = results[name][sample_idx]
            print(f"\n{'─'*40}")
            print(f"【{name}】(passes: {r.get('num_passes', 1)})")
            print(f"{'─'*40}")
            print(r.get("answer", "N/A")[:500])
            if "pass_confidences" in r:
                print(f"\n置信度: {r['pass_confidences']}")
            if "final_action" in r:
                print(f"终止原因: {r['final_action']}")

## 9. 生成对比图表

In [ ]:
from utils.visualization import plot_comparison_chart

# 注：CHAIR 和 POPE 分数需先运行对应评测脚本
# 此处用 timing 数据做示意

names = list(timing.keys())
avg_passes = [timing[n]["avg_passes"] for n in names]

plot_comparison_chart(
    baseline_names=names,
    chair_scores=[0.18, 0.12, 0.07, 0.04],  # 示例数据
    pope_scores=[0.71, 0.74, 0.79, 0.83],   # 示例数据
    avg_passes=avg_passes,
    output_path=f"{REPO_DIR}/results/comparison.png"
)

from IPython.display import Image as IPImage
IPImage(f"{REPO_DIR}/results/comparison.png")

## 10. 保存结果

In [ ]:
import json as json_module
from pathlib import Path

output_dir = Path(REPO_DIR) / "results"
output_dir.mkdir(parents=True, exist_ok=True)

output = {
    "config": {
        "model": MODEL_NAME,
        "num_images": len(images),
        "data_dir": DATA_DIR,
        "question": QUESTION,
    },
    "timing": timing,
}

with open(output_dir / "summary.json", "w") as f:
    json_module.dump(output, f, indent=2, ensure_ascii=False, default=str)

print(f"Results saved to {output_dir / 'summary.json'}")
print(f"Timing summary:")
total_time = sum(t["total_seconds"] for t in timing.values())
print(f"  Total runtime: {total_time:.0f}s ({total_time/60:.1f} min)")
print(f"  Baselines run: {len(timing)}")

---
## 附录：单独测试 IVR 的一次推理

用于快速验证单个样本

In [ ]:
# 快速测试单个样本
test_img = images[0][1] if images else None
if test_img:
    print(f"Testing on: {images[0][0]}")
    
    # B1: 直接回答
    r1 = b1.generate(test_img, QUESTION)
    print(f"\n[B1] Direct (1 pass):")
    print(f"  {r1['answer'][:200]}")
    
    # B4: IVR
    r4 = b4.generate(test_img, QUESTION)
    print(f"\n[B4] IVR ({r4['num_passes']} passes, conf={r4['final_confidence']:.2f}):")
    print(f"  {r4['answer'][:200]}")
    print(f"\n  Pass details:")
    for i, (ans, conf) in enumerate(zip(r4['pass_answers'], r4['pass_confidences'])):
        print(f"    Pass {i+1}: conf={conf:.3f} | {ans[:100]}")